# Getting Started with SEAL

This notebook provides a basic **inference tutorial**:
1. Loading `SEAL` (CONCH backbone) for inference.
2. Loading one HEST Visium sample via `load_hest(...)`.
3. Computing a single patch embedding.
4. Computing whole-slide embedding by mean-pooling patch embeddings.
5. ST spot inference using SEAL-omics


In [ ]:
import os
import time
import shutil
import warnings
from pathlib import Path

import h5py
import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from huggingface_hub import snapshot_download

from hest import load_hest
from seal import seal_factory


if "seal" not in os.listdir():
    os.chdir("..")

print("cwd:", os.getcwd())
print("cuda_available:", torch.cuda.is_available())


## Tutorial Inference Settings

In [ ]:

BACKBONE = "conch"

# HEST dataset settings (no hardcoded machine-specific absolute path)
HEST_REPO_ID = "MahmoodLab/hest"
SAMPLE_ID = "TENX152"  # Edit sample id here
HEST_CACHE_DIR = Path("data/hest_cache")
DOWNLOAD_IF_MISSING = True
PATCH_DIRNAME = "patches_mpp_025"

# Inference settings
PATCH_INDEX = 0
BATCH_SIZE = 64
NUM_WORKERS = 0  # keep 0 for h5py safety in notebooks
MAX_PATCHES = 512  # note: using random subsampling for faster tutorial - set to None to use all patches

# Optional persistence
SAVE_OUTPUTS = True
SAVE_ANNDATA = True
OUTPUT_DIR = Path("tutorials/outputs")

print({
    "BACKBONE": BACKBONE,
    "SAMPLE_ID": SAMPLE_ID,
    "HEST_CACHE_DIR": str(HEST_CACHE_DIR),
    "PATCH_DIRNAME": PATCH_DIRNAME,
})


### HF Authentication checks

In [ ]:
token = os.getenv("HF_TOKEN")
if not token:
    raise RuntimeError("Set HF_TOKEN before running this notebook.")

if not token.startswith("hf_"):
    raise RuntimeError("HF_TOKEN appears invalid (expected prefix 'hf_').")

if shutil.which("hf") is None and shutil.which("huggingface-cli") is None:
    raise RuntimeError(
        'Hugging Face CLI not found. Install with: pip install -U "huggingface_hub[cli]"'
    )

print("HF token set:", True)
print("hf binary:", shutil.which("hf") or shutil.which("huggingface-cli"))


In [ ]:
(img_model, img_transforms, img_precision), gene_model = seal_factory(
    backbone=BACKBONE,
    source="hf",
    hf_token=token,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
img_model = img_model.to(device)
img_model.eval()


In [ ]:
from seal.utils.hest_download import download_hest_sample
hest_dir = download_hest_sample(
    sample_id=SAMPLE_ID,
    local_dir=HEST_CACHE_DIR,
    repo_id=HEST_REPO_ID,
    token=token,
)
sample = load_hest(str(hest_dir), id_list=[SAMPLE_ID])[0]
print("loaded_id:", sample.meta.get("id"))
print("organ:", sample.meta.get("organ"))
print("st_technology:", sample.meta.get("st_technology"))
print("adata_shape:", sample.adata.shape)


## Vision Embeddings

### Preprocess patches 

In [ ]:
patch_dir = Path(hest_dir) / PATCH_DIRNAME
patch_dir.mkdir(parents=True, exist_ok=True)
patch_h5 = patch_dir / f"{SAMPLE_ID}.h5"
if not patch_h5.exists():
    sample.dump_patches(patch_save_dir=patch_dir, name=SAMPLE_ID, target_patch_size=224, target_pixel_size=0.5)
if not patch_h5.exists():
    raise FileNotFoundError(str(patch_h5))
with h5py.File(patch_h5, "r") as f:
    print("patch_h5:", patch_h5)
    print("img_shape:", f["img"].shape)


### Embed single patch

In [ ]:
with h5py.File(patch_h5, "r") as f:
    
    x = img_transforms(Image.fromarray(f["img"][PATCH_INDEX])).unsqueeze(0).to(device)

with torch.inference_mode():
    patch_emb = img_model(x).detach().float().cpu().numpy()
print("single_patch_embedding_shape:", patch_emb.shape)


### Embed whole slide

In [ ]:
with h5py.File(patch_h5, "r") as f:
    imgs = f["img"]
    n = int(imgs.shape[0]) if MAX_PATCHES is None else min(int(imgs.shape[0]), int(MAX_PATCHES))
    embs = []
    
    for i in range(0, n, BATCH_SIZE):
        batch = torch.stack([img_transforms(Image.fromarray(img)) for img in imgs[i:i + BATCH_SIZE]]).to(device)
        with torch.inference_mode():
            embs.append(img_model(batch).detach().float().cpu().numpy())

patch_embeddings = np.concatenate(embs, axis=0)

# when using e.g., MeanPooling
slide_embedding_mean = patch_embeddings.mean(axis=0)

print("patch_embeddings_shape:", patch_embeddings.shape)
print("slide_embedding_mean_shape:", slide_embedding_mean.shape)


In [ ]:
with h5py.File(patch_h5, "r") as f:
    raw_barcodes = f["barcode"][: patch_embeddings.shape[0]].reshape(-1)  # pull barcodes for embedded patches
    coords = f["coords"][: patch_embeddings.shape[0]]  # keep matching coords too

barcodes = np.array([
    b.decode("utf-8") if isinstance(b, (bytes, bytearray)) else str(b)
    for b in raw_barcodes
])  # normalize barcode dtype

obs = sample.adata.obs_names.astype(str)
idx = {k: i for i, k in enumerate(obs)}  # barcode -> adata row
rows = [idx.get(bc) for bc in barcodes]
valid = [i for i, r in enumerate(rows) if r is not None]  # drop barcodes not in adata

aligned = np.full((sample.adata.n_obs, patch_embeddings.shape[1]), np.nan, dtype=np.float32)
aligned[[rows[i] for i in valid]] = patch_embeddings[valid]  # place embeddings into adata row order

sample.adata.obsm["seal_vision_patch_emb"] = aligned
sample.adata.uns["seal_vision_slide_emb_mean"] = slide_embedding_mean.astype(np.float32)

if SAVE_OUTPUTS:
    out_dir = OUTPUT_DIR / SAMPLE_ID
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / "patch_embeddings.npy", patch_embeddings)
    np.save(out_dir / "slide_embedding_mean.npy", slide_embedding_mean)
    np.save(out_dir / "barcodes.npy", barcodes)
    np.save(out_dir / "coords.npy", coords)
    if SAVE_ANNDATA:
        sample.adata.write_h5ad(out_dir / "sample_with_emb.h5ad")  # optional full adata dump
    print("saved_outputs_dir:", out_dir)


## ST Embeddings

In [ ]:
from seal import preprocess_spot

adata_omics, x_gene_np = preprocess_spot(sample, config_path="conf/config.yaml")
x_gene = torch.tensor(x_gene_np, dtype=torch.float32, device=device)

gene_model = gene_model.to(device)
gene_model.eval()
with torch.inference_mode():
    ret = gene_model(x_gene)
z = ret["z"] if isinstance(ret, dict) else ret
omics_emb = z.detach().float().cpu().numpy().astype(np.float32)

adata_omics.obsm["seal_omics_emb"] = omics_emb
print("adata_omics_shape:", adata_omics.shape)
print("omics_emb_shape:", omics_emb.shape)


In [ ]:
if SAVE_OUTPUTS:
    out_dir = OUTPUT_DIR / SAMPLE_ID
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / "omics_embeddings.npy", omics_emb)
    np.save(out_dir / "omics_barcodes.npy", adata_omics.obs_names.astype(str).to_numpy())
    if SAVE_ANNDATA:
        adata_omics.write_h5ad(out_dir / "omics_processed_with_emb.h5ad")
    print("saved_omics_dir:", out_dir)
